In [1]:
import pickle
import glob
import os
from datetime import timedelta
import time
import numpy as np

In [2]:
#!/usr/bin/env python3
"""
Merge all computed similarity blocks into a single file.
Run this after all parallel jobs are complete.
"""




def merge_block_dicts(block_dicts):
    """
    Merge multiple block dictionaries into one.
    
    Args:
        block_dicts: List of result dictionaries from compute_avalanche_similarity_matrix_subset
    
    Returns:
        merged_dict: Single dictionary containing all blocks
    """
    if len(block_dicts) == 0:
        return None
    
    merged = {
        'similarity_blocks': [],
        'path_blocks': [],
        'total_avalanches': block_dicts[0]['total_avalanches'],
        'computed_blocks': []
    }
    
    for block_dict in block_dicts:
        merged['similarity_blocks'].extend(block_dict['similarity_blocks'])
        merged['path_blocks'].extend(block_dict['path_blocks'])
        merged['computed_blocks'].extend(block_dict['computed_blocks'])
    
    return merged


if __name__ == "__main__":
    print("=== MERGING SIMILARITY BLOCKS ===\n")
    
    # Configuration
    input_dir = '/home/novicky/clean_code/similarity_blocks_with_polarity'
    output_file = 'avalanche_similarity_all_blocks_merged.pkl'
    
    # Find all block files
    pattern = os.path.join(input_dir, 'block_*.pkl')
    block_files = sorted(glob.glob(pattern))
    
    if len(block_files) == 0:
        print(f"ERROR: No block files found in {input_dir}/")
        print(f"Looking for pattern: {pattern}")
        exit(1)
    
    print(f"Found {len(block_files)} block files in {input_dir}/")
    print(f"\nFiles to merge:")
    for filename in block_files:
        file_size_mb = os.path.getsize(filename) / (1024 * 1024)
        print(f"  {os.path.basename(filename)} ({file_size_mb:.1f} MB)")
    
    # Load all blocks
    print(f"\nLoading blocks...")
    start_time = time.time()
    all_blocks = []
    
    for i, filename in enumerate(block_files):
        print(f"  Loading {i+1}/{len(block_files)}: {os.path.basename(filename)}")
        with open(filename, 'rb') as f:
            block_dict = pickle.load(f)
            all_blocks.append(block_dict)
    
    load_time = time.time() - start_time
    print(f"Loaded all blocks in {load_time:.1f}s")
    
    # Merge
    print(f"\nMerging blocks...")
    merge_start = time.time()
    merged = merge_block_dicts(all_blocks)
    merge_time = time.time() - merge_start
    print(f"Merged in {merge_time:.1f}s")
    
    # Summary
    print(f"\n{'='*80}")
    print(f"MERGE SUMMARY")
    print(f"{'='*80}")
    print(f"Total blocks merged: {len(merged['similarity_blocks'])}")
    print(f"Total avalanches: {merged['total_avalanches']}")
    print(f"Computed block regions:")
    for block_region in merged['computed_blocks']:
        print(f"  {block_region}")
    
    # Calculate total pairs computed
    total_pairs_computed = sum(
        block['matrix'].size 
        for block in merged['similarity_blocks']
    )
    print(f"\nTotal pairs computed: {total_pairs_computed:,}")
    
    # Save merged result
    print(f"\nSaving merged result to: {output_file}")
    save_start = time.time()
    with open(output_file, 'wb') as f:
        pickle.dump(merged, f)
    save_time = time.time() - save_start
    
    file_size_mb = os.path.getsize(output_file) / (1024 * 1024)
    print(f"Saved in {save_time:.1f}s ({file_size_mb:.1f} MB)")
    
    total_time = time.time() - start_time
    print(f"\n{'='*80}")
    print(f"MERGE COMPLETED")
    print(f"{'='*80}")
    print(f"Total time: {str(timedelta(seconds=int(total_time)))}")
    print(f"Output file: {output_file}")
    print(f"\nYou can now use this file with:")
    print(f"  from your_module import get_distance_from_blocks, get_path_from_blocks")
    print(f"  merged = pickle.load(open('{output_file}', 'rb'))")
    print(f"  dist = get_distance_from_blocks(i, j, merged)")
    print(f"  path = get_path_from_blocks(i, j, merged)")

=== MERGING SIMILARITY BLOCKS ===

Found 15 block files in /home/novicky/clean_code/similarity_blocks_with_polarity/

Files to merge:
  block_000_A0-2044_B0-2044_DIAG.pkl (419.5 MB)
  block_001_A2044-4088_B0-2044.pkl (842.6 MB)
  block_002_A2044-4088_B2044-4088_DIAG.pkl (423.8 MB)
  block_003_A4088-6132_B0-2044.pkl (833.8 MB)
  block_004_A4088-6132_B2044-4088.pkl (838.5 MB)
  block_005_A4088-6132_B4088-6132_DIAG.pkl (416.3 MB)
  block_006_A6132-8176_B0-2044.pkl (839.1 MB)
  block_007_A6132-8176_B2044-4088.pkl (843.7 MB)
  block_008_A6132-8176_B4088-6132.pkl (839.0 MB)
  block_009_A6132-8176_B6132-8176_DIAG.pkl (423.9 MB)
  block_010_A8176-10216_B0-2044.pkl (837.5 MB)
  block_011_A8176-10216_B2044-4088.pkl (842.7 MB)
  block_012_A8176-10216_B4088-6132.pkl (837.7 MB)
  block_013_A8176-10216_B6132-8176.pkl (845.9 MB)
  block_014_A8176-10216_B8176-10216_DIAG.pkl (422.7 MB)

Loading blocks...
  Loading 1/15: block_000_A0-2044_B0-2044_DIAG.pkl
  Loading 2/15: block_001_A2044-4088_B0-2044.pkl

In [3]:

# Initialize matrices
n = merged['total_avalanches']
similarity = np.full((n, n), np.inf, dtype=np.float32)
paths = np.full((n, n), None, dtype=object)

# Fill from blocks
for block in merged['similarity_blocks']:
    row_indices, col_indices = block['row_indices'], block['col_indices']
    similarity[np.ix_(row_indices, col_indices)] = block['matrix']
for block in merged['path_blocks']:
    row_indices, col_indices = block['row_indices'], block['col_indices']
    paths[np.ix_(row_indices, col_indices)] = block['paths']

# Save
pickle.dump(similarity, open('/home/novicky/clean_code/similarity_blocks_with_polarity/similarity_matrix.pkl', 'wb'))
pickle.dump(paths, open('/home/novicky/clean_code/similarity_blocks_with_polarity/path_matrix.pkl', 'wb'))